# 02 — Feature Engineering

**Project:** Supply Chain Analysis and Prediction  

## Purpose
Transform raw data into a model-ready feature matrix — impute, encode, create derived features, and save the processed dataset.

## Flow

| Step | Description |
|------|-------------|
| 1 | Load raw data |
| 2 | Impute missing values |
| 3 | Create engineered features |
| 4 | Outlier analysis (IQR method) |
| 5 | Encode categorical features |
| 6 | Final feature matrix inspection |
| 7 | Save processed data |

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data_loader import load_raw_data
from src.config import IMAGES_DIR, PROCESSED_CSV, PLOT_STYLE, FIGURE_DPI, TARGET_COL

plt.style.use(PLOT_STYLE)
pd.set_option('display.max_columns', None)
print('Setup complete.')

Setup complete.


## 1. Load Raw Data

In [2]:
df = load_raw_data()
print(f'Raw shape: {df.shape}')
display(df.head())

[data_loader] Loaded 50 rows x 10 columns
[data_loader] Columns: ['Product', 'Supplier', 'Warehouse_Location', 'Quantity', 'Unit_Price', 'Total_Cost', 'Delivery_Date', 'Logistics_Partner', 'Shipping_Method', 'Delivery_Status']
Raw shape: (50, 10)


,Product,Supplier,Warehouse_Location,Quantity,Unit_Price,Total_Cost,Delivery_Date,Logistics_Partner,Shipping_Method,Delivery_Status
0,Widget A,Alpha Corp,Warehouse 3,45,16.02,720.90,2025-06-05,FastTrans,Air,Pending
1,Widget A,Alpha Corp,Warehouse 1,37,15.47,572.39,2025-06-20,FastTrans,Road,Pending
2,Widget B,Delta LLC,Warehouse 3,45,41.42,1863.90,2025-06-01,ShipSmart,Sea,Delayed
3,Gadget X,Beta Ltd,Warehouse 1,53,9.60,508.80,2025-06-13,FastTrans,Rail,Delayed
4,Tool Z,Gamma Inc,Warehouse 1,68,29.13,1980.84,2025-06-30,TransEdge,Air,Delayed


## 2. Missing Value Imputation

In [3]:
missing = df.isnull().sum()
print('Missing values per column:')
display(missing[missing > 0].to_frame('count') if missing.sum() > 0 else pd.DataFrame({'Result': ['No missing values — no imputation needed']}))

# No missing values in this dataset, but keeping imputation logic for robustness
for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().any():
        fill_val = df[col].median()
        df[col] = df[col].fillna(fill_val)
        print(f'  Imputed {col} with median={fill_val:.2f}')

for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        fill_val = df[col].mode()[0]
        df[col] = df[col].fillna(fill_val)
        print(f'  Imputed {col} with mode={fill_val}')

print(f'After imputation — missing values: {df.isnull().sum().sum()}')

Missing values per column:


,Result
0,No missing values — no imputation needed


After imputation — missing values: 0


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6328\637351803.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


## 3. Feature Engineering

Creating 6 derived features based on EDA insights:

In [4]:
# ── Feature 1: Delay_Label (target) ──
df['Delay_Label'] = (df['Delivery_Status'] == 'Delayed').astype(int)
print(f'Delay_Label: {df["Delay_Label"].sum()} delayed ({df["Delay_Label"].mean()*100:.1f}%)')

Delay_Label: 16 delayed (32.0%)


In [5]:
# ── Feature 2: Cost_Per_Unit ──
# Captures effective unit economics — different from raw Unit_Price due to bulk discounts
df['Cost_Per_Unit'] = (df['Total_Cost'] / df['Quantity'].replace(0, np.nan)).fillna(0).round(4)
print('Cost_Per_Unit sample:')
display(df[['Quantity', 'Total_Cost', 'Cost_Per_Unit']].head())

Cost_Per_Unit sample:


,Quantity,Total_Cost,Cost_Per_Unit
0,45,720.90,16.02
1,37,572.39,15.47
2,45,1863.90,41.42
3,53,508.80,9.60
4,68,1980.84,29.13


In [6]:
# ── Feature 3: Is_High_Value ──
# Binary flag: items priced above the median are likely more carefully handled
median_price = df['Unit_Price'].median()
df['Is_High_Value'] = (df['Unit_Price'] > median_price).astype(int)
print(f'Median Unit_Price: ${median_price:.2f}')
print(f'High-value items: {df["Is_High_Value"].sum()} ({df["Is_High_Value"].mean()*100:.1f}%)')

Median Unit_Price: $28.19
High-value items: 25 (50.0%)


In [7]:
# ── Features 4-6: Date-based features ──
# Delivery timing can influence delays — month, quarter, day of week
dates = pd.to_datetime(df['Delivery_Date'], errors='coerce')
df['Delivery_Month']     = dates.dt.month
df['Delivery_DayOfWeek'] = dates.dt.dayofweek   # 0=Mon, 6=Sun
df['Delivery_Quarter']   = dates.dt.quarter

print('Date features created:')
display(df[['Delivery_Date', 'Delivery_Month', 'Delivery_DayOfWeek', 'Delivery_Quarter']].head(8))

Date features created:


,Delivery_Date,Delivery_Month,Delivery_DayOfWeek,Delivery_Quarter
0,2025-06-05,6,3,2
1,2025-06-20,6,4,2
2,2025-06-01,6,6,2
3,2025-06-13,6,4,2
4,2025-06-30,6,0,2
5,2025-06-02,6,0,2
6,2025-06-27,6,4,2
7,2025-06-22,6,6,2


In [8]:
# ── Engineered features summary ──
eng_features = pd.DataFrame({
    'Feature': ['Delay_Label', 'Cost_Per_Unit', 'Is_High_Value',
                'Delivery_Month', 'Delivery_DayOfWeek', 'Delivery_Quarter'],
    'Type': ['Binary Target', 'Numeric', 'Binary Flag', 'Numeric', 'Numeric', 'Numeric'],
    'Rationale': [
        'Target: 1=Delayed, 0=On-time/In-Transit/Pending',
        'Total Cost / Quantity — effective unit economics',
        '1 if Unit_Price > median — proxy for item care/priority',
        'Month of delivery — seasonality effects',
        'Day of week — weekday vs weekend delivery patterns',
        'Quarter — Q-end pressure often increases delays'
    ]
})
print('Engineered Features:')
display(eng_features)

Engineered Features:


,Feature,Type,Rationale
0,Delay_Label,Binary Target,"Target: 1=Delayed, 0=On-time/In-Transit/Pending"
1,Cost_Per_Unit,Numeric,Total Cost / Quantity — effective unit economics
2,Is_High_Value,Binary Flag,1 if Unit_Price > median — proxy for item care...
3,Delivery_Month,Numeric,Month of delivery — seasonality effects
4,Delivery_DayOfWeek,Numeric,Day of week — weekday vs weekend delivery patt...
5,Delivery_Quarter,Numeric,Quarter — Q-end pressure often increases delays


In [9]:
# ── Visualize engineered features vs Delay_Label ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Cost_Per_Unit by delay
df.boxplot(column='Cost_Per_Unit', by='Delay_Label', ax=axes[0][0])
axes[0][0].set_title('Cost Per Unit by Delay Label')
axes[0][0].set_xlabel('Delay Label (0=On-time, 1=Delayed)')

# Is_High_Value vs delay rate
hv_delay = df.groupby('Is_High_Value')['Delay_Label'].mean()
hv_delay.plot(kind='bar', ax=axes[0][1], color=['steelblue','tomato'], edgecolor='white')
axes[0][1].set_title('Delay Rate: High Value vs Low Value Items')
axes[0][1].set_xticklabels(['Low Value', 'High Value'], rotation=0)
axes[0][1].set_ylabel('Delay Rate')

# Delay rate by Delivery Month
df.groupby('Delivery_Month')['Delay_Label'].mean().plot(
    kind='bar', ax=axes[0][2], color='darkorange', edgecolor='white')
axes[0][2].set_title('Delay Rate by Delivery Month')
axes[0][2].set_xlabel('Month')
axes[0][2].set_ylabel('Delay Rate')
axes[0][2].tick_params(axis='x', rotation=0)

# Delay rate by Day of Week
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_delay = df.groupby('Delivery_DayOfWeek')['Delay_Label'].mean()
dow_delay.index = [dow_labels[i] for i in dow_delay.index]
dow_delay.plot(kind='bar', ax=axes[1][0], color='mediumseagreen', edgecolor='white')
axes[1][0].set_title('Delay Rate by Day of Week')
axes[1][0].set_ylabel('Delay Rate')
axes[1][0].tick_params(axis='x', rotation=30)

# Delay rate by Quarter
df.groupby('Delivery_Quarter')['Delay_Label'].mean().plot(
    kind='bar', ax=axes[1][1], color='steelblue', edgecolor='white')
axes[1][1].set_title('Delay Rate by Quarter')
axes[1][1].set_xlabel('Quarter')
axes[1][1].set_ylabel('Delay Rate')
axes[1][1].tick_params(axis='x', rotation=0)

# Quantity distribution by delay
df.boxplot(column='Quantity', by='Delay_Label', ax=axes[1][2])
axes[1][2].set_title('Quantity by Delay Label')
axes[1][2].set_xlabel('Delay Label (0=On-time, 1=Delayed)')

plt.suptitle('Engineered Features vs Delay Label', fontsize=14)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '02c_engineered_features.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/02c_engineered_features.png')

Plot saved -> images/02c_engineered_features.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6328\3491423018.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Outlier Analysis (IQR Method)

In [10]:
num_cols = ['Quantity', 'Unit_Price', 'Total_Cost', 'Cost_Per_Unit']

outlier_summary = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({
        'Column': col, 'Q1': round(Q1,2), 'Q3': round(Q3,2),
        'IQR': round(IQR,2), 'Lower Bound': round(lower,2),
        'Upper Bound': round(upper,2), 'Outlier Count': n_outliers,
        'Decision': 'Keep — small dataset (50 rows)' if n_outliers > 0 else 'No outliers'
    })

outlier_df = pd.DataFrame(outlier_summary)
print('Outlier Analysis (IQR Method):')
display(outlier_df)

# Box plots
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[i].set_title(col)
    axes[i].set_ylabel('Value')
fig.suptitle('Outlier Analysis — Box Plots', fontsize=13)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '02d_outlier_boxplots.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/02d_outlier_boxplots.png')
print('\nDecision: Keeping all outliers — dataset is only 50 rows; removing would reduce signal.')

Outlier Analysis (IQR Method):


,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Decision
0,Quantity,34.25,68.00,33.75,-16.38,118.62,0,No outliers
1,Unit_Price,13.56,43.76,30.20,-31.74,89.06,0,No outliers
2,Total_Cost,584.50,1945.16,1360.66,-1456.49,3986.15,4,Keep — small dataset (50 rows)
3,Cost_Per_Unit,13.56,43.76,30.20,-31.74,89.06,0,No outliers


Plot saved -> images/02d_outlier_boxplots.png

Decision: Keeping all outliers — dataset is only 50 rows; removing would reduce signal.


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6328\152919601.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Encode Categorical Features

In [11]:
from sklearn.preprocessing import LabelEncoder

encode_cols = ['Product', 'Supplier', 'Warehouse_Location',
               'Logistics_Partner', 'Shipping_Method', 'Delivery_Status']

le = LabelEncoder()
encoding_map = {}

for col in encode_cols:
    if col in df.columns:
        original_vals = df[col].unique().tolist()
        df[col] = le.fit_transform(df[col].astype(str))
        encoding_map[col] = dict(zip(le.classes_, le.transform(le.classes_)))
        print(f'Encoded {col}: {original_vals} -> {list(encoding_map[col].values())}')

# Drop Delivery_Date (already extracted into month/dow/quarter)
if 'Delivery_Date' in df.columns:
    df.drop(columns=['Delivery_Date'], inplace=True)
    print('Dropped Delivery_Date (date features extracted)')

print(f'\nShape after encoding: {df.shape}')

Encoded Product: ['Widget A', 'Widget B', 'Gadget X', 'Tool Z', 'Gadget Y'] -> [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Encoded Supplier: ['Alpha Corp', 'Delta LLC', 'Beta Ltd', 'Gamma Inc', 'Epsilon Co'] -> [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Encoded Warehouse_Location: ['Warehouse 3', 'Warehouse 1', 'Warehouse 2'] -> [np.int64(0), np.int64(1), np.int64(2)]
Encoded Logistics_Partner: ['FastTrans', 'ShipSmart', 'TransEdge', 'LogiQuick', 'FleetPro'] -> [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Encoded Shipping_Method: ['Air', 'Road', 'Sea', 'Rail'] -> [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Encoded Delivery_Status: ['Pending', 'Delayed', 'In Transit', 'Delivered'] -> [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Dropped Delivery_Date (date features extracted)

Shape after encoding: (50, 15)


## 6. Final Feature Matrix Inspection

In [12]:
print(f'Final shape: {df.shape}')
print(f'All numeric: {df.select_dtypes(include="number").shape[1] == df.shape[1]}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nColumns ({df.shape[1]}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

Final shape: (50, 15)
All numeric: True
Missing values: 0

Columns (15):
   1. Product
   2. Supplier
   3. Warehouse_Location
   4. Quantity
   5. Unit_Price
   6. Total_Cost
   7. Logistics_Partner
   8. Shipping_Method
   9. Delivery_Status
  10. Delay_Label
  11. Cost_Per_Unit
  12. Is_High_Value
  13. Delivery_Month
  14. Delivery_DayOfWeek
  15. Delivery_Quarter


In [13]:
display(df.describe().round(3))

,Product,Supplier,Warehouse_Location,Quantity,Unit_Price,Total_Cost,Logistics_Partner,Shipping_Method,Delivery_Status,Delay_Label,Cost_Per_Unit,Is_High_Value,Delivery_Month,Delivery_DayOfWeek,Delivery_Quarter
count,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.000,50.0,50.000
mean,2.080,1.520,0.860,50.340,28.228,1450.923,1.960,1.580,1.360,0.320,28.228,0.500,6.020,3.2,2.020
std,1.426,1.432,0.833,23.557,14.744,1181.068,1.442,1.071,1.139,0.471,14.744,0.505,0.141,2.1,0.141
min,0.000,0.000,0.000,10.000,5.950,146.400,0.000,0.000,0.000,0.000,5.950,0.000,6.000,0.0,2.000
25%,1.000,0.000,0.000,34.250,13.560,584.498,1.000,1.000,0.000,0.000,13.560,0.000,6.000,2.0,2.000
50%,2.000,1.000,1.000,45.000,28.190,1038.585,2.000,2.000,1.000,0.000,28.190,0.500,6.000,3.5,2.000
75%,3.000,2.000,2.000,68.000,43.760,1945.158,3.000,2.000,2.000,1.000,43.760,1.000,6.000,5.0,2.000
max,4.000,4.000,2.000,93.000,49.930,4583.970,4.000,3.000,3.000,1.000,49.930,1.000,7.000,6.0,3.000


In [14]:
# ── Updated correlation heatmap with engineered features ──
corr = df.corr()
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Full Feature Matrix (Post-Engineering)', fontsize=13)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '02e_full_correlation_heatmap.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/02e_full_correlation_heatmap.png')

Plot saved -> images/02e_full_correlation_heatmap.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6328\1536592035.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Save Processed Data

In [15]:
PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_CSV, index=False)
print(f'Processed data saved -> {PROCESSED_CSV}')
print(f'Shape: {df.shape}')
display(df.head())

Processed data saved -> D:\automation\Supply chain Analysis and Prediction\data\processed\supply_chain_processed.csv
Shape: (50, 15)


,Product,Supplier,Warehouse_Location,Quantity,Unit_Price,Total_Cost,Logistics_Partner,Shipping_Method,Delivery_Status,Delay_Label,Cost_Per_Unit,Is_High_Value,Delivery_Month,Delivery_DayOfWeek,Delivery_Quarter
0,3,0,2,45,16.02,720.90,0,0,3,0,16.02,0,6,3,2
1,3,0,0,37,15.47,572.39,0,2,3,0,15.47,0,6,4,2
2,4,2,2,45,41.42,1863.90,3,3,0,1,41.42,1,6,6,2
3,0,1,0,53,9.60,508.80,0,1,0,1,9.60,0,6,4,2
4,2,4,0,68,29.13,1980.84,4,0,0,1,29.13,1,6,0,2


## Feature Engineering Summary

| Feature | Type | Rationale |
|---|---|---|
| `Delay_Label` | Binary Target | 1=Delayed, 0=On-time/Pending/In-Transit |
| `Cost_Per_Unit` | Numeric | Total Cost / Quantity — effective unit economics |
| `Is_High_Value` | Binary Flag | 1 if Unit_Price > median — proxy for handling priority |
| `Delivery_Month` | Numeric | Month of delivery date — captures seasonality |
| `Delivery_DayOfWeek` | Numeric | 0=Mon to 6=Sun — weekday vs weekend patterns |
| `Delivery_Quarter` | Numeric | Quarter — Q-end rush often increases delays |

**Encoding:** Label encoding applied to Product, Supplier, Warehouse_Location, Logistics_Partner, Shipping_Method, Delivery_Status  
**Outliers:** All retained — dataset is 50 rows; removal would reduce signal significantly  
**Imputation:** Not required — zero missing values in raw data

---

**Next step:** `03_Feature_Selection.ipynb` — LASSO + RFE to select the most predictive features.